In [1]:
%%capture
%pip install -U transformers
#%pip install torch
%pip install -U peft
%pip install -i https://pypi.org/simple/ bitsandbytes
%pip install -U accelerate
%pip install -U trl
%pip install datasets==2.16.0

In [2]:
from warnings import filterwarnings
filterwarnings('ignore')
import json
import re
import wandb
import os
import unicodedata
from pprint import pprint
import pandas as pd
import torch
from kaggle_secrets import UserSecretsClient
from datasets import Dataset, load_dataset
from huggingface_hub import notebook_login, login
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

2024-04-24 15:27:51.530432: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-04-24 15:27:51.530568: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-04-24 15:27:51.653461: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
df = pd.read_csv("/kaggle/input/vietnamese-news-data/Dataset_articles.csv", nrows=5000)
df.head()

,Unnamed: 0,URL,Title,Summary,Contents,Date,Author(s),Category,Tags
0,0,https://laodong.vn/bat-dong-san/thong-tin-ngoc...,"Thông tin “Ngọc Trinh mua đất ở Bảo Lộc"" chỉ l...","Lâm Đồng - Lãnh đạo thành phố Bảo Lộc, Lâm Đồn...","Những ngày vừa qua, trên trang Facebook chính ...","Thứ sáu, 20/05/2022 08:56 (GMT+7)",Phương Nhiên,Bất động sản,"['Lâm Đồng', 'Ngọc Trinh', 'Chiêu trò', 'Giá đ..."
1,1,https://laodong.vn/bat-dong-san/lo-hong-trong-...,Lỗ hổng trong việc thẩm tra năng lực tài chính...,TPHCM - Việc không thể cưỡng chế thuế của hai ...,"Theo thông tin từ Cục Thuế TP.HCM, hiện cơ qua...","Thứ sáu, 20/05/2022 08:10 (GMT+7)",Gia Miêu,Bất động sản,"['Thủ Thiêm', 'Đấu giá đất']"
2,2,https://laodong.vn/bat-dong-san/som-hoan-thien...,Sớm hoàn thiện các dự án nhà ở xã hội để CNLĐ ...,"Hiện trên địa bàn tỉnh Ninh Bình có 32 khu, cụ...",CNLĐ mong muốn sớm được tiếp cận với nhà ở xã ...,"Thứ sáu, 20/05/2022 07:47 (GMT+7)",NGUYỄN TRƯỜNG,Bất động sản,"['Dự án', 'Nhà ở xã hội', 'Dự án nhà ở xã hội'..."
3,3,https://laodong.vn/bat-dong-san/chi-tiet-ho-so...,Chi tiết hồ sơ hoàn công nhà ở năm 2022,Hoàn công nhà ở với ý nghĩa là điều kiện để đư...,Hoàn công nhà ở là một thủ tục hành chính tron...,"Thứ sáu, 20/05/2022 06:44 (GMT+7)",Kim Nhung (T/H),Bất động sản,"['Giấy phép xây dựng', 'Hồ sơ hoàn công', 'nhà..."
4,4,https://laodong.vn/bat-dong-san/khoi-tao-khong...,"Khởi tạo không gian sống đẳng cấp, đón sóng đầ...",Có rất nhiều lý do khiến những dự án thấp nội ...,Đi dọc đường Lê Văn Lương kéo dài xuống khu Dư...,"Thứ năm, 19/05/2022 15:30 (GMT+7)",Huyền Nguyễn,Bất động sản,['An Quý Villa']


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  5000 non-null   int64 
 1   URL         5000 non-null   object
 2   Title       5000 non-null   object
 3   Summary     5000 non-null   object
 4   Contents    4918 non-null   object
 5   Date        5000 non-null   object
 6   Author(s)   5000 non-null   object
 7   Category    5000 non-null   object
 8   Tags        5000 non-null   object
dtypes: int64(1), object(8)
memory usage: 351.7+ KB


In [5]:
df.dropna(inplace=True)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4918 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  4918 non-null   int64 
 1   URL         4918 non-null   object
 2   Title       4918 non-null   object
 3   Summary     4918 non-null   object
 4   Contents    4918 non-null   object
 5   Date        4918 non-null   object
 6   Author(s)   4918 non-null   object
 7   Category    4918 non-null   object
 8   Tags        4918 non-null   object
dtypes: int64(1), object(8)
memory usage: 384.2+ KB


In [7]:
df[['Category']].stack().unique()

array(['Bất động sản', 'Xã hội', 'Lao Động cuối tuần', 'Kinh doanh',
       'Thời sự', 'Công đoàn', 'Bạn đọc', 'Pháp luật'], dtype=object)

In [8]:
len(df)

4918

In [9]:
print(df['Summary'].to_list()[0])


Lâm Đồng - Lãnh đạo thành phố Bảo Lộc, Lâm Đồng khẳng định thông tin người mẫu Ngọc Trinh mua 11ha đất ở địa phương để làm homestay là hoàn toàn sai sự thật. Đó chỉ là chiêu trò của giới bất động sản để thổi giá đất.


In [10]:
df1 = df[['Summary', 'Contents']]

In [11]:
df1.iloc[0]["Summary"], df1.iloc[0]["Contents"]

('Lâm\xa0Đồng -\xa0Lãnh đạo thành phố Bảo Lộc, Lâm Đồng\xa0khẳng định thông\xa0tin\xa0người mẫu Ngọc Trinh mua 11ha đất ở địa phương để làm homestay là\xa0hoàn toàn sai sự thật.\xa0Đó\xa0chỉ là chiêu trò của\xa0giới bất động sản\xa0để thổi giá đất.',
 'Những ngày\xa0vừa\xa0qua, trên\xa0trang Facebook chính chủ của Ngọc Trinh có đăng tải thông tin người mẫu này vừa mua\xa011ha đất với homestay\xa0tại xã Đại Lào, TP.Bảo Lộc, thu hút sự chú ý lớn từ đông đảo cộng đồng mạng.\xa0 Sau\xa0đó, một số trang thông tin và mạng\xa0xã hội khác tiếp tục đăng tại lại thông tin này,\xa0đồng thời khẳng định, khu đất Ngọc Trinh mới mua nằm tại xã Đại Lào, TP.Bảo Lộc,\xa0tọa lạc ở vị trí đắc địa, tiếp giáp núi và có khí hậu mát mẻ dễ chịu, thích hợp cho kỳ nghỉ trốn nắng hè cho du khách. Trước\xa0thông tin này, ông Đoàn Kim Đình - Chủ tịch UBND thành phố Bảo Lộc cho\xa0biết,\xa0UBND thành phố đã yêu cầu cơ quan chức năng và UBND xã Đại Lào phối hợp kiểm chứng thông tin. Qua xác minh,\xa0khu đất Ngọc Trin

In [12]:
dataset = Dataset.from_pandas(df1)
dataset

Dataset({
    features: ['Summary', 'Contents', '__index_level_0__'],
    num_rows: 4918
})

### Formatted Prompt

Bạn là một trợ lí Tiếng Việt nhiệt tình và trung thực. Hãy luôn trả lời một cách hữu ích nhất có thể, đồng thời giữ an toàn.
Câu trả lời của bạn không nên chứa bất kỳ nội dung gây hại, phân biệt chủng tộc, phân biệt giới tính, độc hại, nguy hiểm hoặc bất hợp pháp nào. Hãy đảm bảo rằng các câu trả lời của bạn không có thiên kiến xã hội và mang tính tích cực.Nếu một câu hỏi không có ý nghĩa hoặc không hợp lý về mặt thông tin, hãy giải thích tại sao thay vì trả lời một điều gì đó không chính xác. Nếu bạn không biết câu trả lời cho một câu hỏi, hãy trẳ lời là bạn không biết và vui lòng không chia sẻ thông tin sai lệch.


{prompt} 



In [13]:
DEFAULT_SYSTEM_PROMPT = """
Bạn là một trợ lí Tiếng Việt nhiệt tình và trung thực. Hãy luôn trả lời một cách hữu ích nhất có thể, đồng thời giữ an toàn.
Câu trả lời của bạn không nên chứa bất kỳ nội dung gây hại, phân biệt chủng tộc, phân biệt giới tính, độc hại, nguy hiểm hoặc bất hợp pháp nào. 
Hãy đảm bảo rằng các câu trả lời của bạn không có thiên kiến xã hội và mang tính tích cực. Nếu một câu hỏi không có ý nghĩa hoặc không hợp lý về 
mặt thông tin, hãy giải thích tại sao thay vì trả lời một điều gì đó không chính xác. Nếu bạn không biết câu trả lời cho một câu hỏi, hãy trẳ lời 
là bạn không biết và vui lòng không chia sẻ thông tin sai lệch. 
Dưới đây là một tin tức. Bạn hãy viết một đoạn tóm tắt tin tức trên trong khoảng 100 từ. Hãy tóm tắt nội dung chính, quan trọng của tin tức đó, 
đồng thời đoạn tóm tắt phải được viết lại rõ ràng, thông tin được viết lại hợp lý. 
 
"""

In [14]:
def generate_training_prompt(
    article: str, summary: str, system_prompt: str = DEFAULT_SYSTEM_PROMPT
) -> str:
    
    input_prefix = "<s>[INST] "
    input_suffix = "[/INST] "
    pre_prompt_prefix = "<s>[INST]  <<SYS>>\n"
    pre_prompt_suffix = "<</SYS>> \n\n"
    eos_token = "</s>"
    
    full_prompt = ""
    full_prompt += pre_prompt_prefix
    full_prompt += "\n" + system_prompt
    full_prompt += pre_prompt_suffix
    full_prompt += "\n" + article
    full_prompt += input_suffix
    full_prompt += "\n\n### Response:"
    full_prompt += "\n" + summary
    full_prompt += eos_token
    return full_prompt

In [15]:
# Split the processed dataset into train, validation, and test sets
train_dataset = dataset.shuffle(seed=21).select(range(0, int(0.8 * len(dataset))))
validation_dataset = dataset.shuffle(seed=21).select(range(int(0.8 * len(dataset)), int(0.9 * len(dataset))))
test_dataset = dataset.shuffle(seed=21).select(range(int(0.9 * len(dataset)), len(dataset)))

In [16]:
!pip install py-vncorenlp

  Preparing metadata (setup.py) ... - \ done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 6.0 MB/s eta 0:00:00
  Created wheel for py-vncorenlp: filename=py_vncorenlp-0.1.4-py3-none-any.whl size=4305 sha256=3d166a9d7a13482cf0838f82da35629326bff7b1f02d028b684f41d4353f4d2e
  Stored in directory: /root/.cache/pip/wheels/d5/d9/bf/62632cdb007c702a0664091e92a0bb1f18a2fcecbe962d9827
Successfully built py-vncorenlp


In [17]:
!sudo apt -y install default-jdk




The following additional packages will be installed:
  default-jdk-headless libice-dev libpthread-stubs0-dev libsm-dev libx11-dev
  libxau-dev libxcb1-dev libxdmcp-dev libxt-dev openjdk-11-jdk
  openjdk-11-jdk-headless x11proto-core-dev x11proto-dev xorg-sgml-doctools
  xtrans-dev
Suggested packages:
  libice-doc libsm-doc libx11-doc libxcb-doc libxt-doc openjdk-11-demo
  openjdk-11-source visualvm
The following NEW packages will be installed:
  default-jdk default-jdk-headless libice-dev libpthread-stubs0-dev libsm-dev
  libx11-dev libxau-dev libxcb1-dev libxdmcp-dev libxt-dev openjdk-11-jdk
  openjdk-11-jdk-headless x11proto-core-dev x11proto-dev xorg-sgml-doctools
  xtrans-dev
0 upgraded, 16 newly installed, 0 to remove and 65 not upgraded.
Need to get 76.9 MB of archives.
After this operation, 90.6 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu focal-updates/main amd64 openjdk-11-jdk-headless amd64 11.0.22+7-0ubuntu2~20.04.1 [73.7 MB]
Get:2 http

In [18]:
!javac -version

javac 11.0.22


In [19]:
import py_vncorenlp
py_vncorenlp.download_model(save_dir='/kaggle/working/')


--2024-04-24 15:28:32--  https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/VnCoreNLP-1.2.jar
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 27412703 (26M) [application/octet-stream]
Saving to: 'VnCoreNLP-1.2.jar'

     0K .......... .......... .......... .......... ..........  0% 2.38M 11s
    50K .......... .......... .......... .......... ..........  0% 13.8M 6s
   100K .......... .......... .......... .......... ..........  0% 6.55M 6s
   150K .......... .......... .......... .......... ..........  0% 23.8M 4s
   200K .......... .......... .......... .......... ..........  0% 23.1M 4s
   250K .......... .......... .......... .......... ..........  1% 41.7M 3s
   300K .......... .......... .......... .......... ..........  1% 9.28M 3s
   350K .

In [20]:
rdrsegmenter = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir='/kaggle/working/')

2024-04-24 15:28:37 INFO  WordSegmenter:24 - Loading Word Segmentation model


In [21]:
def preprocess_data(data_point):
    tokenized_input = ' '.join(rdrsegmenter.word_segment(' '.join([data_point['Contents'], data_point['input']])))

    model_input = tokenizer(tokenized_input, max_length=max_source_length, padding=False, truncation=True)

    # Tokenize targets with the `text_target` keyword argument
    tokenized_target = ' '.join(rdrsegmenter.word_segment(data_point['output']))
    labels = tokenizer(text_target=tokenized_target, max_length=max_target_length, padding=False, truncation=True)

    # If we are padding here, replace all tokenizer.pad_token_id in the labels by -100 when we want to ignore
    # padding in the loss.
    # if padding == "max_length" and data_args.ignore_pad_token_for_loss:
    #     labels["input_ids"] = [
    #         [(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels["input_ids"]
    #     ]

    model_input["labels"] = labels["input_ids"]
    return model_input

In [22]:
def create_prompt(data_point):
    article, summary = data_point["Contents"], data_point["Summary"]
#     article = unicodedata.normalize("utf8", article)
    article = re.sub(r'\s(\.)\s', ' ', article)
    
    summary = re.sub(r'([^a-zA-Z\s])?\n(\w+)', r'\1 \2', summary)
    summary = re.sub(r'\s(\.)', '.', summary)
    return article.strip(), summary.strip()


def generate_text(data_point):
    article, summary = create_prompt(data_point)
    return generate_training_prompt(article, summary)

In [23]:
example_data_point = {
    "Contents": df.iloc[5]["Contents"],
    "Summary": df.iloc[5]["Summary"]
}


example = generate_text(example_data_point)
print(example)

<s>[INST]  <<SYS>>


Bạn là một trợ lí Tiếng Việt nhiệt tình và trung thực. Hãy luôn trả lời một cách hữu ích nhất có thể, đồng thời giữ an toàn.
Câu trả lời của bạn không nên chứa bất kỳ nội dung gây hại, phân biệt chủng tộc, phân biệt giới tính, độc hại, nguy hiểm hoặc bất hợp pháp nào. 
Hãy đảm bảo rằng các câu trả lời của bạn không có thiên kiến xã hội và mang tính tích cực. Nếu một câu hỏi không có ý nghĩa hoặc không hợp lý về 
mặt thông tin, hãy giải thích tại sao thay vì trả lời một điều gì đó không chính xác. Nếu bạn không biết câu trả lời cho một câu hỏi, hãy trẳ lời 
là bạn không biết và vui lòng không chia sẻ thông tin sai lệch. 
Dưới đây là một tin tức. Bạn hãy viết một đoạn tóm tắt tin tức trên trong khoảng 100 từ. Hãy tóm tắt nội dung chính, quan trọng của tin tức đó, 
đồng thời đoạn tóm tắt phải được viết lại rõ ràng, thông tin được viết lại hợp lý. 
 
<</SYS>> 


Những ngày qua, đất khu đô thị Thanh Hà Cienco đang có hiện tượng các cò đất thổi giá khi có thông tin chủ đ

In [24]:
MODEL_NAME = "Viet-Mistral/Vistral-7B-Chat"
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

In [25]:
def create_model_and_tokenizer():
    nf4_config = BitsAndBytesConfig(
        load_in_4bit = True,
        bnb_4bit_quant_type = "nf4",
        bnb_4bit_use_double_quant = True,
        bnb_4bit_compute_dtype = torch.bfloat16
    )


    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map='auto',
        quantization_config=nf4_config,
        use_cache=False
    )
    
    model.config.use_cache = False
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    return model, tokenizer

In [26]:
import os

os.environ["WANDB_PROJECT"] = "vietnamese-news-summarization"
os.environ["WANDB_LOG_MODEL"] = "checkpoint"

In [ ]:
import wandb

wandb.login()
wandb.init()

wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: vu25ck28. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.16.6
wandb: Run data is saved locally in /kaggle/working/wandb/run-20240424_152839-8h0k61vj
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run fast-night-2
wandb: ⭐️ View project at https://wandb.ai/vu25ck28/vietnamese-news-summarization
wandb: 🚀 View run at https://wandb.ai/vu25ck28/vietnamese-news-summarization/runs/8h0k61vj


In [ ]:
from huggingface_hub import login

login()

Token has not been saved to git credential helper. Pass `add_to_git_credential=True` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /root/.cache/huggingface/token
Login successful


In [29]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

model, tokenizer = create_model_and_tokenizer()

config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/10.0G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.59G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer.model:   0%|          | 0.00/597k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.15M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/169 [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [30]:
if torch.cuda.device_count() > 1: # If more than 1 GPU
    model.is_parallelizable = True
    model.model_parallel = True
model.config.quantization_config.to_dict()

{'quant_method': <QuantizationMethod.BITS_AND_BYTES: 'bitsandbytes'>,
 '_load_in_8bit': False,
 '_load_in_4bit': True,
 'llm_int8_threshold': 6.0,
 'llm_int8_skip_modules': None,
 'llm_int8_enable_fp32_cpu_offload': False,
 'llm_int8_has_fp16_weight': False,
 'bnb_4bit_quant_type': 'nf4',
 'bnb_4bit_use_double_quant': True,
 'bnb_4bit_compute_dtype': 'bfloat16',
 'bnb_4bit_quant_storage': 'uint8',
 'load_in_4bit': True,
 'load_in_8bit': False}

In [31]:
# Set LoRA configuration
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
        "lm_head",
    ],
    bias="none",
    task_type="CAUSAL_LM",
)

In [32]:
from peft import *

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)


In [33]:
##############################
# TrainingArguments parameters
##############################

# Output directory where the model predictions and checkpoints will be stored
output_dir = "vietnamese-news-summarization-vistral-7b-finetuned"

# Number of training epochs
# num_train_epochs = 5

# Enable fp16/bf16 training (set bf16 to True with an A100)
fp16 = False
bf16 = True

In [34]:
training_arguments = TrainingArguments(
    per_device_train_batch_size = 4,
    logging_steps = 10,
    learning_rate = 2e-4,
    max_steps = 100, # the total number of training steps to perform
    num_train_epochs = 3,
    evaluation_strategy="steps",
    eval_steps=20,
    save_steps=20,
    warmup_steps = 0.03,
    save_strategy="steps",
    group_by_length = True,
    output_dir = output_dir,
    report_to="wandb",  # Set report_to here
    run_name='finetune-vistral-7b',
    save_safetensors=True,
    lr_scheduler_type = 'constant',
    load_best_model_at_end = True,
    push_to_hub = True,
    push_to_hub_model_id = "vietnamese-news-summarization-vistral-7b",
    push_to_hub_token=access_token_write
)

In [35]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    peft_config=peft_config,
    formatting_func=generate_text,
    max_seq_length=300,
    tokenizer=tokenizer,
    args=training_arguments,
    packing=True
)

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

max_steps is given, it will override any value given in num_train_epochs


In [36]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

with torch.no_grad():
    torch.cuda.empty_cache()

# Fine-tune your model
trainer.train()

Step,Training Loss,Validation Loss
20,1.230600,1.291836
40,1.280200,1.265140
60,1.408400,1.252776
80,1.294400,1.249703
100,1.288000,1.253894


wandb: Adding directory to artifact (./vietnamese-news-summarization-vistral-7b-finetuned/checkpoint-20)... Done. 15.6s
wandb: Adding directory to artifact (./vietnamese-news-summarization-vistral-7b-finetuned/checkpoint-40)... Done. 19.7s
wandb: Adding directory to artifact (./vietnamese-news-summarization-vistral-7b-finetuned/checkpoint-60)... Done. 17.7s
wandb: Adding directory to artifact (./vietnamese-news-summarization-vistral-7b-finetuned/checkpoint-80)... Done. 21.2s
wandb: Adding directory to artifact (./vietnamese-news-summarization-vistral-7b-finetuned/checkpoint-100)... Done. 25.3s
max_steps is given, it will override any value given in num_train_epochs


TrainOutput(global_step=100, training_loss=1.3397535133361815, metrics={'train_runtime': 12863.9065, 'train_samples_per_second': 0.031, 'train_steps_per_second': 0.008, 'total_flos': 5261210956800000.0, 'train_loss': 1.3397535133361815, 'epoch': 0.02976190476190476})

In [37]:
trainer.push_to_hub()

CommitInfo(commit_url='https://huggingface.co/anhvu2501/vietnamese-news-summarization-vistral-7b/commit/33cd19fb8af1eeae5591916f1fe843db74a637b0', commit_message='End of training', commit_description='', oid='33cd19fb8af1eeae5591916f1fe843db74a637b0', pr_url=None, pr_revision=None, pr_num=None)

In [38]:
trainer.save_model("vietnamese-news-summarization-finetuned-vistral-7b")

### Inference

In [39]:
gc.collect()
torch.cuda.empty_cache()

In [40]:
artifact = wandb.Api().artifact("vu25ck28/vietnamese-news-summarization/checkpoint-robust-night-1:v0", type='model')
artifact.download()

wandb: Downloading large artifact checkpoint-robust-night-1:v0, 2553.69MB. 13 files... 
wandb:   13 of 13 files downloaded.  
Done. 0:0:22.7


'/kaggle/working/artifacts/checkpoint-robust-night-1:v0'

In [41]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

nf4_config = BitsAndBytesConfig(
        load_in_4bit = True,
        bnb_4bit_quant_type = "nf4",
        bnb_4bit_use_double_quant = True,
        bnb_4bit_compute_dtype = torch.bfloat16
    )

inf_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map='auto',
        quantization_config=nf4_config,
        use_cache=False
    )

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [42]:
model = PeftModel.from_pretrained(inf_model, "/kaggle/working/artifacts/checkpoint-robust-night-1:v0")

In [43]:
test_dataset['Contents'][1]

'Thế hệ “không xu dính túi” đang thay đổi ngành bất động sản Tập 5 series talk show "NGƯỜI TIÊN PHONG" mới lên sóng với chủ đề "Gen Z: Phá vỡ giới hạn - Đột phá tiên phong" đã vẽ nên chân dung thế hệ Gen Z điển hình - những người đang buộc thị trường phải thay đổi theo hướng tích cực hơn. Vân và Thái là đại diện cho thế hệ những người trẻ thành công sớm. Sinh ra trong thời kỳ đất nước hội nhập, tiếp thu những công nghệ mới, tư duy về tài sản cùng phong cách đầu tư xuống tiền của Gen Z đã táo bạo hơn trước rất nhiều, ngay cả trong lĩnh vực đòi hỏi sự đầu tư, tích lũy và chuẩn bị trong một thời gian dài đối với các thế hệ trước như mua nhà. Ngôi nhà chính là minh chứng cho khả năng kiếm tiền và là một trong những thước đo cho sự thành công của giới trẻ toàn cầu. Thay vì chọn nhà là nơi để ở như nếp nghĩ của các thế hệ trước, Gen Z coi nhà là một khoản đầu tư. Hai khách mời đồng tình rằng, với nhà ở, Gen Z không chỉ mưu cầu những khoản lãi về tiền bạc, mà họ còn tìm kiếm trải nghiệm, sức 

In [44]:
from tqdm import tqdm

In [45]:
prompt="""<s>[INST]  <<SYS>>


Bạn là một trợ lí Tiếng Việt nhiệt tình và trung thực. Hãy luôn trả lời một cách hữu ích nhất có thể, đồng thời giữ an toàn.
Câu trả lời của bạn không nên chứa bất kỳ nội dung gây hại, phân biệt chủng tộc, phân biệt giới tính, độc hại, nguy hiểm hoặc bất hợp pháp nào. 
Hãy đảm bảo rằng các câu trả lời của bạn không có thiên kiến xã hội và mang tính tích cực. Nếu một câu hỏi không có ý nghĩa hoặc không hợp lý về 
mặt thông tin, hãy giải thích tại sao thay vì trả lời một điều gì đó không chính xác. Nếu bạn không biết câu trả lời cho một câu hỏi, hãy trẳ lời 
là bạn không biết và vui lòng không chia sẻ thông tin sai lệch. 
Dưới đây là một tin tức. Bạn hãy viết một đoạn tóm tắt tin tức trên trong khoảng 100 từ. Hãy tóm tắt nội dung chính, quan trọng của tin tức đó, 
đồng thời đoạn tóm tắt phải được viết lại rõ ràng, thông tin được viết lại hợp lý. 
 
<</SYS>> 

Thế hệ “không xu dính túi” đang thay đổi ngành bất động sản Tập 5 series talk show "NGƯỜI TIÊN PHONG" mới lên sóng với chủ đề "Gen Z: Phá vỡ giới hạn - 
Đột phá tiên phong" đã vẽ nên chân dung thế hệ Gen Z điển hình - những người đang buộc thị trường phải thay đổi theo hướng tích cực hơn. Vân và Thái 
là đại diện cho thế hệ những người trẻ thành công sớm. Sinh ra trong thời kỳ đất nước hội nhập, tiếp thu những công nghệ mới, tư duy về tài sản cùng 
phong cách đầu tư xuống tiền của Gen Z đã táo bạo hơn trước rất nhiều, ngay cả trong lĩnh vực đòi hỏi sự đầu tư, tích lũy và chuẩn bị trong một 
thời gian dài đối với các thế hệ trước như mua nhà. Ngôi nhà chính là minh chứng cho khả năng kiếm tiền và là một trong những thước đo cho sự 
thành công của giới trẻ toàn cầu. Thay vì chọn nhà là nơi để ở như nếp nghĩ của các thế hệ trước, Gen Z coi nhà là một khoản đầu tư. Hai khách mời 
đồng tình rằng, với nhà ở, Gen Z không chỉ mưu cầu những khoản lãi về tiền bạc, mà họ còn tìm kiếm trải nghiệm, sức khỏe, chất lượng sống. 

### Response:
"""
device = "cuda" if torch.cuda.is_available() else "cpu"
model_input = tokenizer(prompt, return_tensors="pt").to(device)


_ = model.eval()
with torch.no_grad():
    out = model.generate(**model_input, max_new_tokens=200)

print(tokenizer.decode(out[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[INST]   <<SYS>> 


Bạn là một trợ lí Tiếng Việt nhiệt tình và trung thực. Hãy luôn trả lời một cách hữu ích nhất có thể, đồng thời giữ an toàn.
Câu trả lời của bạn không nên chứa bất kỳ nội dung gây hại, phân biệt chủng tộc, phân biệt giới tính, độc hại, nguy hiểm hoặc bất hợp pháp nào. 
Hãy đảm bảo rằng các câu trả lời của bạn không có thiên kiến xã hội và mang tính tích cực. Nếu một câu hỏi không có ý nghĩa hoặc không hợp lý về 
mặt thông tin, hãy giải thích tại sao thay vì trả lời một điều gì đó không chính xác. Nếu bạn không biết câu trả lời cho một câu hỏi, hãy trẳ lời 
là bạn không biết và vui lòng không chia sẻ thông tin sai lệch. 
Dưới đây là một tin tức. Bạn hãy viết một đoạn tóm tắt tin tức trên trong khoảng 100 từ. Hãy tóm tắt nội dung chính, quan trọng của tin tức đó, 
đồng thời đoạn tóm tắt phải được viết lại rõ ràng, thông tin được viết lại hợp lý. 
 
<</SYS>>  

Thế hệ “không xu dính túi” đang thay đổi ngành bất động sản Tập 5 series talk show "NGƯỜI TIÊN PHONG" mới lên